In [ ]:
!nvidia-smi

Thu Mar  5 20:24:53 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [ ]:
# python3.12

In [ ]:
pip install wheel setuptools

In [ ]:
pip install torch==2.10.0 torchvision==0.25.0

Looking in indexes: https://download.pytorch.org/whl/cu130


In [ ]:
pip install timm 'accelerate>=0.26.0' qwen-vl-utils==0.0.14 transformers==5.0.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 MB 55.6 MB/s eta 0:00:00


In [ ]:
pip install ms-swift==3.9.0

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.3/43.3 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.8/298.8 kB 10.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 12.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.6/449.6 kB 26.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 876.5/876.5 kB 44.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.1/6.1 MB 133.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 504.9/504.9 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 122.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Inference

In [ ]:
# pip install pypdfium2

In [ ]:
# pip install vllm==0.11.0

In [ ]:
# !unzip -qq OL_dataset.zip

In [ ]:
from pathlib import Path
import json
import re
from pypdfium2 import PdfDocument

IMAGE_IE_PROMPT_TEMPLATE = """Тебе будет дано изображение страницы  PDF из опросного листа (технического документа на оборудование).
Извлеки спсиок характеристик и требований в JSON формате. Без комментариев, без пояснений, без Markdown.

JSON format:
[
  {
    "name": "<название параметра, которое запрашивает Заказчик>",
    "request_value": "<значение парметра, которое запрашивает Заказчик>"
  },
  ...
]

СТРОГИЕ ПРАВИЛА:

1) Верни ТОЛЬКО JSON. Любой текст вне JSON запрещён.

2) name:
   - Используй формулировку характеристики или требования из документа.
   - Нормализуй переносы строк и пробелы, но НЕ меняй смысл.
   - Если требование оформлено предложением (например: "… должен …"), используй ВЕСЬ текст требования как name, в одну строку.

3) request_value:
   - Для параметров, заданных в таблицах или структурированных блоках, указывай значение, сопоставленное этому параметру в документе.
   - Для декларативных требований, не имеющих явного значения, используй подтверждающее значение "Да".
   - Значения вида:
       "Определяет завод-изготовитель",
       "Уточняется согласно рабочей документации",
       "По таблице …",
       "___", "____"
     считаются валидными значениями и должны сохраняться без изменений.
   - Если параметр содержит несколько подзначений, объединяй их в ОДНУ строку через "; ", сохраняя все числа, единицы измерения и ссылки.
   - Для формы с чек-боксами укажи только выбранные варианты

4) НЕ добавляй атрибуты, которых нет в документе.
5) НЕ изменяй единицы измерения, знаки «−», «+», десятичные разделители.
6) Игнорируй титульный лист, подписи, реквизиты, контактную информацию — извлекай только технические характеристики и требования.
7) Не оставляй несколько требований в одном. Если они оформлены в одной ячейке таблицы или одним абзацем, разбей их в отдельные.
8) Верни все характеристики из текста страниц, даже исли они представленны частично.
9) Игнорируй справочные таблицы в приложениях, со сводными значениями/прогнозами без требований или характеристик относящихся к какой-либо продукции.
"""

# Inpud PDFs dir
pdf_dir = Path("./OL_dataset")

# Dir for converted dataset
out_dir = Path("./qwen_infer_dataset")
scale = 3.0

# qwen smart_resize reference params
IMAGE_FACTOR = 32
MIN_TOKENS = 4
MAX_TOKENS = 2150

SYSTEM_PROMPT_RU = (
    "Ты точно извлекаешь пары ключ-значение из технических документов. "
    "Возвращай только валидный JSON без пояснений."
)
USER_PROMPT = IMAGE_IE_PROMPT_TEMPLATE.strip() + "\n\n<image>"


def convert_pdf_to_images(pdf_path, scale=3.0):
    pdf_document = PdfDocument(pdf_path.as_posix())
    pages = []
    for page_index in range(len(pdf_document)):
        page = pdf_document[page_index]
        pil_image = page.render(scale=scale).to_pil()
        pages.append(pil_image)
        page.close()
    pdf_document.close()
    return pages

def round_by_factor(number: int, factor: int) -> int:
    return round(number / factor) * factor

def ceil_by_factor(number: int, factor: int) -> int:
    import math
    return math.ceil(number / factor) * factor

def floor_by_factor(number: int, factor: int) -> int:
    import math
    return math.floor(number / factor) * factor

def smart_resize(height: int, width: int, min_pixels=MIN_TOKENS, max_pixels=MAX_TOKENS, factor=IMAGE_FACTOR):
    import math
    MAX_RATIO = 200
    if max(height, width) / min(height, width) > MAX_RATIO:
        raise ValueError("absolute aspect ratio must be smaller than 200")
    min_area = min_pixels * factor * factor
    max_area = max_pixels * factor * factor
    h_bar = max(factor, round_by_factor(height, factor))
    w_bar = max(factor, round_by_factor(width, factor))
    if h_bar * w_bar > max_area:
        beta = math.sqrt((height * width) / max_area)
        h_bar = floor_by_factor(height / beta, factor)
        w_bar = floor_by_factor(width / beta, factor)
    elif h_bar * w_bar < min_area:
        beta = math.sqrt(min_area / (height * width))
        h_bar = ceil_by_factor(height * beta, factor)
        w_bar = ceil_by_factor(width * beta, factor)
    return h_bar, w_bar


images_out = out_dir / "images"
images_out.mkdir(parents=True, exist_ok=True)
jsonl_path = out_dir / "dataset.jsonl"

rows = 0
with jsonl_path.open("w", encoding="utf-8") as f:
    for pdf_path in sorted(pdf_dir.glob("*.pdf")):
        pdf_tag = pdf_path.stem
        pages = convert_pdf_to_images(pdf_path, scale=scale)

        for page_idx, img in enumerate(pages):
            w, h = img.size
            new_h, new_w = smart_resize(height=h, width=w, min_pixels=MIN_TOKENS, max_pixels=MAX_TOKENS, factor=IMAGE_FACTOR)
            img = img.convert("RGB").resize((new_w, new_h))

            img_name = f"image_{pdf_tag}_p{page_idx}.png"
            img_path = images_out / img_name
            img.save(img_path, format="PNG")

            item = {
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT_RU},
                    {"role": "user", "content": USER_PROMPT},
                ],
                "images": [img_path.as_posix()],
            }
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
            rows += 1

print(f"Done. Rows: {rows}")
print(f"JSONL: {jsonl_path}")
print(f"Images dir: {images_out}")


Done. Rows: 117
JSONL: qwen_infer_dataset/dataset.jsonl
Images dir: qwen_infer_dataset/images


In [ ]:

model_chkp = Path('/content/output_sft/v0-20260305-203024/checkpoint-50')
dataset_path = Path('/content/qwen_infer_dataset/dataset.jsonl')
results_path = Path('./OL_dataset_results_nb2_t07.jsonl')

In [ ]:
!CUDA_VISIBLE_DEVICES=0 \
VLLM_ALLOW_LONG_MAX_MODEL_LEN=1 \
swift infer \
    --adapters {model_chkp.absolute().as_posix()} \
    --model Qwen/Qwen3-VL-8B-Instruct \
    --merge_lora True \
    --infer_backend vllm \
    --temperature 0.7 \
    --num_beams 2 \
    --torch_dtype bfloat16 \
    --vllm_gpu_memory_utilization 0.9 \
    --vllm_max_model_len 16000 \
    --val_dataset {dataset_path.absolute().as_posix()} \
    --max_new_tokens 5000 \
    --result_path {results_path.absolute().as_posix()} \
    --use_hf=1


run sh: `/usr/bin/python3 /usr/local/lib/python3.12/dist-packages/swift/cli/infer.py --adapters /content/output_sft/v0-20260305-203024/checkpoint-50 --model Qwen/Qwen3-VL-8B-Instruct --merge_lora True --infer_backend vllm --temperature 0.3 --num_beams 1 --torch_dtype bfloat16 --vllm_gpu_memory_utilization 0.9 --vllm_max_model_len 16000 --val_dataset /content/qwen_infer_dataset/dataset.jsonl --max_new_tokens 5000 --result_path /content/OL_dataset_results_nb1_t03.jsonl --use_hf=1`
2026-03-05 21:32:22.336222: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-05 21:32:22.355980: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:

In [ ]:
import json
import pandas as pd
from pathlib import Path

df=pd.read_json('./OL_dataset_results_nb2_t07_8b.jsonl', lines=True)
df['extracted'] = df['response'].apply(json.loads)
df['doc_name'] = (
    df['images']
    .apply(lambda x: x[0]['path'])
    .apply(lambda x: x.split('_')[-2])
)
df['page'] = (
    df['images']
    .apply(lambda x: x[0]['path'])
    .apply(lambda x: x.split('_')[-1])
    .apply(lambda x: x.replace('.png' ,'').replace('p' ,''))
    .astype(int)
)

results_dir = Path('./OL_dataset/')
for doc_name in results_dir.glob('*.pdf'):
    extracted = df.loc[df['doc_name'] == doc_name.stem, 'extracted'].tolist()

    extracted = [kv for page in extracted for kv in page]
    store_path = results_dir / (doc_name.stem + '_qwen-8b-ft.json')
    with open(store_path, 'w') as f:
        f.write(json.dumps(extracted, ensure_ascii=False))

